In [18]:
import stim
import numpy as np
import itertools
import pymatching

def generate_3_round_data_only_circuit(rotation_indices, rotation_angle_rad=0.0):
    """
    Generates a full 3-round surface code circuit (Round 0, 1, 2) + Final Readout.
    Restricts active coherent rotation injection STRICTLY to data qubits during Round 1.
    """
    circuit = stim.Circuit()

    # --- QUBIT REGISTERS MAPPING ---
    def cq(i): return i
    def zq(i): return 9 + i
    def xq(i): return 13 + i
    all_xq = [xq(0), xq(1), xq(2), xq(3)]

    # Structural helper for the uniform syndrome extraction timeline
    def apply_syndrome_cycle(c):
        c.append("H", all_xq)
        
        # CNOT Layers 1-4
        c.append("CX", [cq(1), zq(1),  cq(3), zq(2),  cq(5), zq(3)])
        c.append("CX", [xq(1), cq(0),  xq(2), cq(4),  xq(3), cq(6)])
        c.append("CX", [cq(4), zq(1),  cq(6), zq(2),  cq(8), zq(3)])
        c.append("CX", [xq(1), cq(1),  xq(2), cq(5),  xq(3), cq(7)])
        
        # CNOT Layers 5-8
        c.append("CX", [cq(0), zq(0),  cq(2), zq(1),  cq(4), zq(2)])
        c.append("CX", [xq(0), cq(1),  xq(1), cq(3),  xq(2), cq(7)])
        c.append("CX", [cq(3), zq(0),  cq(5), zq(1),  cq(7), zq(2)])
        c.append("CX", [xq(0), cq(2),  xq(1), cq(4),  xq(2), cq(8)])
        c.append("H", all_xq)

    meas_targets = [xq(0), zq(0), xq(1), zq(1), xq(2), zq(2), xq(3), zq(3)]
    
    # ----------------------------------------------------
    # ROUND 0: Stable Baseline Initialization
    # ----------------------------------------------------
    apply_syndrome_cycle(circuit)
    circuit.append("M", meas_targets)
    circuit.append("R", meas_targets)
    
    # Initial deterministic Z-basis anchors
    circuit.append("DETECTOR", stim.target_rec(-7))  # zq0
    circuit.append("DETECTOR", stim.target_rec(-5))  # zq1
    circuit.append("DETECTOR", stim.target_rec(-3))  # zq2
    circuit.append("DETECTOR", stim.target_rec(-1))  # zq3

    # ----------------------------------------------------
    # ROUND 1: COHERENT QUBIT ROTATIONS INJECTED HERE (X-ERROR)
    # ----------------------------------------------------
    if rotation_indices and rotation_angle_rad > 0.0:
        # Calculate Born Rule projection probability: P = sin^2(theta / 2)
        projection_prob = np.sin(rotation_angle_rad / 2.0) ** 2
        circuit.append("X_ERROR", [cq(i) for i in rotation_indices], projection_prob)
        
    apply_syndrome_cycle(circuit)
    circuit.append("M", meas_targets)
    circuit.append("R", meas_targets)
    
    # Natively tracks ancilla fluctuations relative to Round 0
    for i in range(8):
        circuit.append("DETECTOR", [stim.target_rec(-8 + i), stim.target_rec(-16 + i)])

    # ----------------------------------------------------
    # ROUND 2: Verification Buffer Round
    # ----------------------------------------------------
    apply_syndrome_cycle(circuit)
    circuit.append("M", meas_targets)
    
    # Tracks stabilization relative to Round 1
    for i in range(8):
        circuit.append("DETECTOR", [stim.target_rec(-8 + i), stim.target_rec(-16 + i)])

    # ----------------------------------------------------
    # FINAL READOUT DATA DESTRUCTION BOUNDARY
    # ----------------------------------------------------
    data_targets = [cq(i) for i in range(9)]
    circuit.append("M", data_targets)

    # Closes out the final stabilizer tracking frame (Data vs Round 2 Z-ancillas)
    circuit.append("DETECTOR", [stim.target_rec(-9), stim.target_rec(-6), stim.target_rec(-16)])  # zq0
    circuit.append("DETECTOR", [stim.target_rec(-8), stim.target_rec(-7), stim.target_rec(-5), stim.target_rec(-4), stim.target_rec(-14)])  # zq1
    circuit.append("DETECTOR", [stim.target_rec(-6), stim.target_rec(-5), stim.target_rec(-3), stim.target_rec(-2), stim.target_rec(-12)])  # zq2
    circuit.append("DETECTOR", [stim.target_rec(-4), stim.target_rec(-1), stim.target_rec(-10)])  # zq3

    circuit.append("OBSERVABLE_INCLUDE", [stim.target_rec(-9), stim.target_rec(-8), stim.target_rec(-7)], 0)

    return circuit

# --- MAIN EXECUTION BLOCK ---
if __name__ == "__main__":
    print("--- 3-Round Data-Only X-Error Rotations - Incremental Sweep ---")
    
    num_data_qubits = 9
    shots_per_test = 1000  # Multi-shot sampling tracks rotation projection rates cleanly
    
    # Generate the linear range from 0 rad to 2*pi with 15 steps
    rotation_angles = np.linspace(0, 2 * np.pi, 16)[1:]
    
    # Initialize a stable baseline graph topology map for PyMatching
    base_circuit = generate_3_round_data_only_circuit(rotation_indices=list(range(num_data_qubits)), rotation_angle_rad=0.5)
    error_model = base_circuit.detector_error_model(decompose_errors=True)
    matching = pymatching.Matching.from_detector_error_model(error_model)
    
    # Sweep through the incremental steps
    for angle_rad in rotation_angles:
        deg = np.degrees(angle_rad)
        proj_p = np.sin(angle_rad / 2.0) ** 2
        print(f"\n=========================================================================")
        print(f"Testing Rotation Angle: {angle_rad:.2f} rad ({deg:.1f}°) | Projection Prob: {proj_p*100:.2f}%")
        print(f"=========================================================================")
        
        # Inner loop sweeps over dimension weights to observe scaling behaviors per angle slice
        for weight in range(1, num_data_qubits + 1):
            combinations = list(itertools.combinations(range(num_data_qubits), weight))
            total_failures_in_dimension = 0
            total_shots_in_dimension = len(combinations) * shots_per_test
            
            for rotation_set in combinations:
                test_circuit = generate_3_round_data_only_circuit(rotation_indices=rotation_set, rotation_angle_rad=angle_rad)
                
                sampler = test_circuit.compile_detector_sampler()
                det_samples, obs_samples = sampler.sample(shots=shots_per_test, separate_observables=True)
                
                predicted_observables = matching.decode_batch(det_samples)
                actual_observables = obs_samples[:, 0]
                
                shot_failures = np.sum(actual_observables != predicted_observables[:, 0])
                total_failures_in_dimension += shot_failures
                    
            logical_error_rate = (total_failures_in_dimension / total_shots_in_dimension) * 100
            print(f"  Dimension {weight} (Size {len(combinations)}): Logical Error Rate = {logical_error_rate:.2f}%")


--- 3-Round Data-Only X-Error Rotations - Incremental Sweep ---

Testing Rotation Angle: 0.42 rad (24.0°) | Projection Prob: 4.32%
  Dimension 1 (Size 9): Logical Error Rate = 0.00%
  Dimension 2 (Size 36): Logical Error Rate = 0.07%
  Dimension 3 (Size 84): Logical Error Rate = 0.24%
  Dimension 4 (Size 126): Logical Error Rate = 0.55%
  Dimension 5 (Size 126): Logical Error Rate = 0.89%
  Dimension 6 (Size 84): Logical Error Rate = 1.32%
  Dimension 7 (Size 36): Logical Error Rate = 1.62%
  Dimension 8 (Size 9): Logical Error Rate = 2.36%
  Dimension 9 (Size 1): Logical Error Rate = 3.00%

Testing Rotation Angle: 0.84 rad (48.0°) | Projection Prob: 16.54%
  Dimension 1 (Size 9): Logical Error Rate = 0.00%
  Dimension 2 (Size 36): Logical Error Rate = 1.44%
  Dimension 3 (Size 84): Logical Error Rate = 3.77%
  Dimension 4 (Size 126): Logical Error Rate = 6.82%
  Dimension 5 (Size 126): Logical Error Rate = 10.25%
  Dimension 6 (Size 84): Logical Error Rate = 13.97%
  Dimension 7 (Size

In [19]:
import stim
import numpy as np
import itertools
import pymatching

def generate_3_round_data_only_circuit(rotation_indices, rotation_angle_rad=0.0):
    """
    Generates a full 3-round surface code circuit (Round 0, 1, 2) + Final Readout.
    Restricts active coherent rotation injection STRICTLY to data qubits during Round 1.
    """
    circuit = stim.Circuit()

    # --- QUBIT REGISTERS MAPPING ---
    def cq(i): return i
    def zq(i): return 9 + i
    def xq(i): return 13 + i
    all_xq = [xq(0), xq(1), xq(2), xq(3)]

    # Structural helper for the uniform syndrome extraction timeline
    def apply_syndrome_cycle(c):
        c.append("H", all_xq)
        
        # CNOT Layers 1-4
        c.append("CX", [cq(1), zq(1),  cq(3), zq(2),  cq(5), zq(3)])
        c.append("CX", [xq(1), cq(0),  xq(2), cq(4),  xq(3), cq(6)])
        c.append("CX", [cq(4), zq(1),  cq(6), zq(2),  cq(8), zq(3)])
        c.append("CX", [xq(1), cq(1),  xq(2), cq(5),  xq(3), cq(7)])
        
        # CNOT Layers 5-8
        c.append("CX", [cq(0), zq(0),  cq(2), zq(1),  cq(4), zq(2)])
        c.append("CX", [xq(0), cq(1),  xq(1), cq(3),  xq(2), cq(7)])
        c.append("CX", [cq(3), zq(0),  cq(5), zq(1),  cq(7), zq(2)])
        c.append("CX", [xq(0), cq(2),  xq(1), cq(4),  xq(2), cq(8)])
        c.append("H", all_xq)

    meas_targets = [xq(0), zq(0), xq(1), zq(1), xq(2), zq(2), xq(3), zq(3)]
    
    # ----------------------------------------------------
    # ROUND 0: Stable Baseline Initialization
    # ----------------------------------------------------
    apply_syndrome_cycle(circuit)
    circuit.append("M", meas_targets)
    circuit.append("R", meas_targets)
    
    # Initial deterministic Z-basis anchors
    circuit.append("DETECTOR", stim.target_rec(-7))  # zq0
    circuit.append("DETECTOR", stim.target_rec(-5))  # zq1
    circuit.append("DETECTOR", stim.target_rec(-3))  # zq2
    circuit.append("DETECTOR", stim.target_rec(-1))  # zq3

    # ----------------------------------------------------
    # ROUND 1: COHERENT QUBIT ROTATIONS INJECTED HERE (Z-ERROR)
    # ----------------------------------------------------
    if rotation_indices and rotation_angle_rad > 0.0:
        # Calculate Born Rule projection probability: P = sin^2(theta / 2)
        projection_prob = np.sin(rotation_angle_rad / 2.0) ** 2
        circuit.append("Z_ERROR", [cq(i) for i in rotation_indices], projection_prob)
        
    apply_syndrome_cycle(circuit)
    circuit.append("M", meas_targets)
    circuit.append("R", meas_targets)
    
    # Natively tracks ancilla fluctuations relative to Round 0
    for i in range(8):
        circuit.append("DETECTOR", [stim.target_rec(-8 + i), stim.target_rec(-16 + i)])

    # ----------------------------------------------------
    # ROUND 2: Verification Buffer Round
    # ----------------------------------------------------
    apply_syndrome_cycle(circuit)
    circuit.append("M", meas_targets)
    
    # Tracks stabilization relative to Round 1
    for i in range(8):
        circuit.append("DETECTOR", [stim.target_rec(-8 + i), stim.target_rec(-16 + i)])

    # ----------------------------------------------------
    # FINAL READOUT DATA DESTRUCTION BOUNDARY
    # ----------------------------------------------------
    data_targets = [cq(i) for i in range(9)]
    circuit.append("M", data_targets)

    # Closes out the final stabilizer tracking frame (Data vs Round 2 Z-ancillas)
    circuit.append("DETECTOR", [stim.target_rec(-9), stim.target_rec(-6), stim.target_rec(-16)])  # zq0
    circuit.append("DETECTOR", [stim.target_rec(-8), stim.target_rec(-7), stim.target_rec(-5), stim.target_rec(-4), stim.target_rec(-14)])  # zq1
    circuit.append("DETECTOR", [stim.target_rec(-6), stim.target_rec(-5), stim.target_rec(-3), stim.target_rec(-2), stim.target_rec(-12)])  # zq2
    circuit.append("DETECTOR", [stim.target_rec(-4), stim.target_rec(-1), stim.target_rec(-10)])  # zq3

    circuit.append("OBSERVABLE_INCLUDE", [stim.target_rec(-9), stim.target_rec(-8), stim.target_rec(-7)], 0)

    return circuit

# --- MAIN EXECUTION BLOCK ---
if __name__ == "__main__":
    print("--- 3-Round Data-Only Z-Error Rotations - Incremental Sweep ---")
    
    num_data_qubits = 9
    shots_per_test = 1000  # Multi-shot sampling tracks rotation projection rates cleanly
    
    # Generate the linear range from 0 rad to 2*pi with 15 steps
    rotation_angles = np.linspace(0, 2 * np.pi, 16)[1:]
    
    # Initialize a stable baseline graph topology map for PyMatching
    base_circuit = generate_3_round_data_only_circuit(rotation_indices=list(range(num_data_qubits)), rotation_angle_rad=0.5)
    error_model = base_circuit.detector_error_model(decompose_errors=True)
    matching = pymatching.Matching.from_detector_error_model(error_model)
    
    # Sweep through the incremental steps
    for angle_rad in rotation_angles:
        deg = np.degrees(angle_rad)
        proj_p = np.sin(angle_rad / 2.0) ** 2
        print(f"\n=========================================================================")
        print(f"Testing Rotation Angle: {angle_rad:.2f} rad ({deg:.1f}°) | Projection Prob: {proj_p*100:.2f}%")
        print(f"=========================================================================")
        
        # Inner loop sweeps over dimension weights to observe scaling behaviors per angle slice
        for weight in range(1, num_data_qubits + 1):
            combinations = list(itertools.combinations(range(num_data_qubits), weight))
            total_failures_in_dimension = 0
            total_shots_in_dimension = len(combinations) * shots_per_test
            
            for rotation_set in combinations:
                test_circuit = generate_3_round_data_only_circuit(rotation_indices=rotation_set, rotation_angle_rad=angle_rad)
                
                sampler = test_circuit.compile_detector_sampler()
                det_samples, obs_samples = sampler.sample(shots=shots_per_test, separate_observables=True)
                
                predicted_observables = matching.decode_batch(det_samples)
                actual_observables = obs_samples[:, 0]
                
                shot_failures = np.sum(actual_observables != predicted_observables[:, 0])
                total_failures_in_dimension += shot_failures
                    
            logical_error_rate = (total_failures_in_dimension / total_shots_in_dimension) * 100
            print(f"  Dimension {weight} (Size {len(combinations)}): Logical Error Rate = {logical_error_rate:.2f}%")


--- 3-Round Data-Only Z-Error Rotations - Incremental Sweep ---

Testing Rotation Angle: 0.42 rad (24.0°) | Projection Prob: 4.32%
  Dimension 1 (Size 9): Logical Error Rate = 0.00%
  Dimension 2 (Size 36): Logical Error Rate = 0.00%
  Dimension 3 (Size 84): Logical Error Rate = 0.00%
  Dimension 4 (Size 126): Logical Error Rate = 0.00%
  Dimension 5 (Size 126): Logical Error Rate = 0.00%
  Dimension 6 (Size 84): Logical Error Rate = 0.00%
  Dimension 7 (Size 36): Logical Error Rate = 0.00%
  Dimension 8 (Size 9): Logical Error Rate = 0.00%
  Dimension 9 (Size 1): Logical Error Rate = 0.00%

Testing Rotation Angle: 0.84 rad (48.0°) | Projection Prob: 16.54%
  Dimension 1 (Size 9): Logical Error Rate = 0.00%
  Dimension 2 (Size 36): Logical Error Rate = 0.00%
  Dimension 3 (Size 84): Logical Error Rate = 0.00%
  Dimension 4 (Size 126): Logical Error Rate = 0.00%
  Dimension 5 (Size 126): Logical Error Rate = 0.00%
  Dimension 6 (Size 84): Logical Error Rate = 0.00%
  Dimension 7 (Size 3

In [22]:
import stim
import numpy as np
import itertools
import pymatching

def generate_3_round_data_only_circuit(rotation_indices, rotation_angle_rad=0.0):
    """
    Generates a full 3-round surface code circuit (Round 0, 1, 2) + Final Readout.
    Restricts active coherent rotation injection STRICTLY to data qubits during Round 1.
    """
    circuit = stim.Circuit()

    # --- QUBIT REGISTERS MAPPING ---
    def cq(i): return i
    def zq(i): return 9 + i
    def xq(i): return 13 + i
    all_xq = [xq(0), xq(1), xq(2), xq(3)]

    # Structural helper for the uniform syndrome extraction timeline
    def apply_syndrome_cycle(c):
        c.append("H", all_xq)
        
        # CNOT Layers 1-4
        c.append("CX", [cq(1), zq(1),  cq(3), zq(2),  cq(5), zq(3)])
        c.append("CX", [xq(1), cq(0),  xq(2), cq(4),  xq(3), cq(6)])
        c.append("CX", [cq(4), zq(1),  cq(6), zq(2),  cq(8), zq(3)])
        c.append("CX", [xq(1), cq(1),  xq(2), cq(5),  xq(3), cq(7)])
        
        # CNOT Layers 5-8
        c.append("CX", [cq(0), zq(0),  cq(2), zq(1),  cq(4), zq(2)])
        c.append("CX", [xq(0), cq(1),  xq(1), cq(3),  xq(2), cq(7)])
        c.append("CX", [cq(3), zq(0),  cq(5), zq(1),  cq(7), zq(2)])
        c.append("CX", [xq(0), cq(2),  xq(1), cq(4),  xq(2), cq(8)])
        c.append("H", all_xq)

    meas_targets = [xq(0), zq(0), xq(1), zq(1), xq(2), zq(2), xq(3), zq(3)]
    
    # ----------------------------------------------------
    # ROUND 0: Stable Baseline Initialization
    # ----------------------------------------------------
    apply_syndrome_cycle(circuit)
    circuit.append("M", meas_targets)
    circuit.append("R", meas_targets)
    
    # Initial deterministic Z-basis anchors
    circuit.append("DETECTOR", stim.target_rec(-7))  # zq0
    circuit.append("DETECTOR", stim.target_rec(-5))  # zq1
    circuit.append("DETECTOR", stim.target_rec(-3))  # zq2
    circuit.append("DETECTOR", stim.target_rec(-1))  # zq3

    # ----------------------------------------------------
    # ROUND 1: COHERENT QUBIT ROTATIONS INJECTED HERE (Y-ERROR)
    # ----------------------------------------------------
    if rotation_indices and rotation_angle_rad > 0.0:
        # Calculate Born Rule projection probability: P = sin^2(theta / 2)
        projection_prob = np.sin(rotation_angle_rad / 2.0) ** 2
        circuit.append("Y_ERROR", [cq(i) for i in rotation_indices], projection_prob)
        
    apply_syndrome_cycle(circuit)
    circuit.append("M", meas_targets)
    circuit.append("R", meas_targets)
    
    # Natively tracks ancilla fluctuations relative to Round 0
    for i in range(8):
        circuit.append("DETECTOR", [stim.target_rec(-8 + i), stim.target_rec(-16 + i)])

    # ----------------------------------------------------
    # ROUND 2: Verification Buffer Round
    # ----------------------------------------------------
    apply_syndrome_cycle(circuit)
    circuit.append("M", meas_targets)
    
    # Tracks stabilization relative to Round 1
    for i in range(8):
        circuit.append("DETECTOR", [stim.target_rec(-8 + i), stim.target_rec(-16 + i)])

    # ----------------------------------------------------
    # FINAL READOUT DATA DESTRUCTION BOUNDARY
    # ----------------------------------------------------
    data_targets = [cq(i) for i in range(9)]
    circuit.append("M", data_targets)

    # Closes out the final stabilizer tracking frame (Data vs Round 2 Z-ancillas)
    circuit.append("DETECTOR", [stim.target_rec(-9), stim.target_rec(-6), stim.target_rec(-16)])  # zq0
    circuit.append("DETECTOR", [stim.target_rec(-8), stim.target_rec(-7), stim.target_rec(-5), stim.target_rec(-4), stim.target_rec(-14)])  # zq1
    circuit.append("DETECTOR", [stim.target_rec(-6), stim.target_rec(-5), stim.target_rec(-3), stim.target_rec(-2), stim.target_rec(-12)])  # zq2
    circuit.append("DETECTOR", [stim.target_rec(-4), stim.target_rec(-1), stim.target_rec(-10)])  # zq3

    circuit.append("OBSERVABLE_INCLUDE", [stim.target_rec(-9), stim.target_rec(-8), stim.target_rec(-7)], 0)

    return circuit

# --- MAIN EXECUTION BLOCK ---
if __name__ == "__main__":
    print("--- 3-Round Data-Only Y-Error Rotations - Incremental Sweep ---")
    
    num_data_qubits = 9
    shots_per_test = 1000  # Multi-shot sampling tracks rotation projection rates cleanly
    
    # Generate the linear range from 0 rad to 2*pi with 15 steps
    rotation_angles = np.linspace(0, 2 * np.pi, 16)[1:]
    
    # Initialize a stable baseline graph topology map for PyMatching
    base_circuit = generate_3_round_data_only_circuit(rotation_indices=list(range(num_data_qubits)), rotation_angle_rad=0.5)
    error_model = base_circuit.detector_error_model(decompose_errors=True, ignore_decomposition_failures=True)
    matching = pymatching.Matching.from_detector_error_model(error_model)
    
    # Sweep through the incremental steps
    for angle_rad in rotation_angles:
        deg = np.degrees(angle_rad)
        proj_p = np.sin(angle_rad / 2.0) ** 2
        print(f"\n=========================================================================")
        print(f"Testing Rotation Angle: {angle_rad:.2f} rad ({deg:.1f}°) | Projection Prob: {proj_p*100:.2f}%")
        print(f"=========================================================================")
        
        # Inner loop sweeps over dimension weights to observe scaling behaviors per angle slice
        for weight in range(1, num_data_qubits + 1):
            combinations = list(itertools.combinations(range(num_data_qubits), weight))
            total_failures_in_dimension = 0
            total_shots_in_dimension = len(combinations) * shots_per_test
            
            for rotation_set in combinations:
                test_circuit = generate_3_round_data_only_circuit(rotation_indices=rotation_set, rotation_angle_rad=angle_rad)
                
                sampler = test_circuit.compile_detector_sampler()
                det_samples, obs_samples = sampler.sample(shots=shots_per_test, separate_observables=True)
                
                predicted_observables = matching.decode_batch(det_samples)
                actual_observables = obs_samples[:, 0]
                
                shot_failures = np.sum(actual_observables != predicted_observables[:, 0])
                total_failures_in_dimension += shot_failures
                    
            logical_error_rate = (total_failures_in_dimension / total_shots_in_dimension) * 100
            print(f"  Dimension {weight} (Size {len(combinations)}): Logical Error Rate = {logical_error_rate:.2f}%")


--- 3-Round Data-Only Y-Error Rotations - Incremental Sweep ---

Testing Rotation Angle: 0.42 rad (24.0°) | Projection Prob: 4.32%
  Dimension 1 (Size 9): Logical Error Rate = 0.49%
  Dimension 2 (Size 36): Logical Error Rate = 1.00%
  Dimension 3 (Size 84): Logical Error Rate = 1.49%
  Dimension 4 (Size 126): Logical Error Rate = 1.90%
  Dimension 5 (Size 126): Logical Error Rate = 2.39%
  Dimension 6 (Size 84): Logical Error Rate = 2.91%
  Dimension 7 (Size 36): Logical Error Rate = 3.50%
  Dimension 8 (Size 9): Logical Error Rate = 4.01%
  Dimension 9 (Size 1): Logical Error Rate = 4.10%

Testing Rotation Angle: 0.84 rad (48.0°) | Projection Prob: 16.54%
  Dimension 1 (Size 9): Logical Error Rate = 1.90%
  Dimension 2 (Size 36): Logical Error Rate = 3.83%
  Dimension 3 (Size 84): Logical Error Rate = 5.55%
  Dimension 4 (Size 126): Logical Error Rate = 7.29%
  Dimension 5 (Size 126): Logical Error Rate = 9.17%
  Dimension 6 (Size 84): Logical Error Rate = 11.06%
  Dimension 7 (Size 

In [24]:
import stim
import numpy as np
import itertools
import pymatching

def generate_3_round_all_qubits_circuit(rotation_indices, rotation_angle_rad=0.0):
    """
    Generates a full 3-round surface code circuit (Round 0, 1, 2) + Final Readout.
    Accepts raw integer indices to inject coherent rotations across ANY data or ancilla qubits.
    """
    circuit = stim.Circuit()

    # --- QUBIT REGISTERS MAPPING ---
    def cq(i): return i
    def zq(i): return 9 + i
    def xq(i): return 13 + i
    all_xq = [xq(0), xq(1), xq(2), xq(3)]

    # Structural helper for the uniform syndrome extraction timeline
    def apply_syndrome_cycle(c):
        c.append("H", all_xq)
        
        # CNOT Layers 1-4
        c.append("CX", [cq(1), zq(1),  cq(3), zq(2),  cq(5), zq(3)])
        c.append("CX", [xq(1), cq(0),  xq(2), cq(4),  xq(3), cq(6)])
        c.append("CX", [cq(4), zq(1),  cq(6), zq(2),  cq(8), zq(3)])
        c.append("CX", [xq(1), cq(1),  xq(2), cq(5),  xq(3), cq(7)])
        
        # CNOT Layers 5-8
        c.append("CX", [cq(0), zq(0),  cq(2), zq(1),  cq(4), zq(2)])
        c.append("CX", [xq(0), cq(1),  xq(1), cq(3),  xq(2), cq(7)])
        c.append("CX", [cq(3), zq(0),  cq(5), zq(1),  cq(7), zq(2)])
        c.append("CX", [xq(0), cq(2),  xq(1), cq(4),  xq(2), cq(8)])
        c.append("H", all_xq)

    meas_targets = [xq(0), zq(0), xq(1), zq(1), xq(2), zq(2), xq(3), zq(3)]
    
    # ----------------------------------------------------
    # ROUND 0: Stable Baseline Initialization
    # ----------------------------------------------------
    apply_syndrome_cycle(circuit)
    circuit.append("M", meas_targets)
    circuit.append("R", meas_targets)
    
    # Initial deterministic Z-basis anchors
    circuit.append("DETECTOR", stim.target_rec(-7))  # zq0
    circuit.append("DETECTOR", stim.target_rec(-5))  # zq1
    circuit.append("DETECTOR", stim.target_rec(-3))  # zq2
    circuit.append("DETECTOR", stim.target_rec(-1))  # zq3

    # ----------------------------------------------------
    # ROUND 1: GLOBAL COHERENT ROTATIONS (DATA + ANCILLA) (X-ERROR)
    # ----------------------------------------------------
    if rotation_indices and rotation_angle_rad > 0.0:
        # Calculate Born Rule projection probability: P = sin^2(theta / 2)
        projection_prob = np.sin(rotation_angle_rad / 2.0) ** 2
        # Target the raw input integer array directly
        circuit.append("X_ERROR", rotation_indices, projection_prob)
        
    apply_syndrome_cycle(circuit)
    circuit.append("M", meas_targets)
    circuit.append("R", meas_targets)
    
    # Natively tracks ancilla fluctuations relative to Round 0
    for i in range(8):
        circuit.append("DETECTOR", [stim.target_rec(-8 + i), stim.target_rec(-16 + i)])

    # ----------------------------------------------------
    # ROUND 2: Verification Buffer Round
    # ----------------------------------------------------
    apply_syndrome_cycle(circuit)
    circuit.append("M", meas_targets)
    
    # Tracks stabilization relative to Round 1
    for i in range(8):
        circuit.append("DETECTOR", [stim.target_rec(-8 + i), stim.target_rec(-16 + i)])

    # ----------------------------------------------------
    # FINAL READOUT DATA DESTRUCTION BOUNDARY
    # ----------------------------------------------------
    data_targets = [cq(i) for i in range(9)]
    circuit.append("M", data_targets)

    # Closes out the final stabilizer tracking frame (Data vs Round 2 Z-ancillas)
    circuit.append("DETECTOR", [stim.target_rec(-9), stim.target_rec(-6), stim.target_rec(-16)])  # zq0
    circuit.append("DETECTOR", [stim.target_rec(-8), stim.target_rec(-7), stim.target_rec(-5), stim.target_rec(-4), stim.target_rec(-14)])  # zq1
    circuit.append("DETECTOR", [stim.target_rec(-6), stim.target_rec(-5), stim.target_rec(-3), stim.target_rec(-2), stim.target_rec(-12)])  # zq2
    circuit.append("DETECTOR", [stim.target_rec(-4), stim.target_rec(-1), stim.target_rec(-10)])  # zq3

    circuit.append("OBSERVABLE_INCLUDE", [stim.target_rec(-9), stim.target_rec(-8), stim.target_rec(-7)], 0)

    return circuit

# --- MAIN EXECUTION BLOCK ---
if __name__ == "__main__":
    print("--- 3-Round Global (Data + Ancillas) X-Error Rotations ---")
    
    # Total physical index map array: 9 data qubits + 4 Z-ancillas + 4 X-ancillas = 17 qubits total
    total_circuit_qubits = 17 
    shots_per_test = 1000 
    
    # Incremental steps sweep list configuration
    rotation_angles = np.linspace(0, 2 * np.pi, 16)[1:]
    
    # Initialize graph topology on ALL 17 elements to open edge definitions inside PyMatching
    base_circuit = generate_3_round_all_qubits_circuit(rotation_indices=list(range(total_circuit_qubits)), rotation_angle_rad=0.5)
    error_model = base_circuit.detector_error_model(decompose_errors=True)
    matching = pymatching.Matching.from_detector_error_model(error_model)
    
    # Multi-Angle Sweep Controller
    for angle_rad in rotation_angles:
        deg = np.degrees(angle_rad)
        proj_p = np.sin(angle_rad / 2.0) ** 2
        print(f"\n=========================================================================")
        print(f"Testing Rotation Angle: {angle_rad:.2f} rad ({deg:.1f}°) | Projection Prob: {proj_p*100:.2f}%")
        print(f"=========================================================================")
        
        # Inner dimension sweep expands to handle up to the full 17 element set bounds
        for weight in range(1, total_circuit_qubits + 1):
            combinations = list(itertools.combinations(range(total_circuit_qubits), weight))
            total_failures_in_dimension = 0
            total_shots_in_dimension = len(combinations) * shots_per_test
            
            for rotation_set in combinations:
                # Convert tuple permutation block cleanly into raw integer list array for Stim
                test_circuit = generate_3_round_all_qubits_circuit(rotation_indices=list(rotation_set), rotation_angle_rad=angle_rad)
                
                sampler = test_circuit.compile_detector_sampler()
                det_samples, obs_samples = sampler.sample(shots=shots_per_test, separate_observables=True)
                
                predicted_observables = matching.decode_batch(det_samples)
                actual_observables = obs_samples[:, 0]
                
                shot_failures = np.sum(actual_observables != predicted_observables[:, 0])
                total_failures_in_dimension += shot_failures
                    
            logical_error_rate = (total_failures_in_dimension / total_shots_in_dimension) * 100
            print(f"  Dimension {weight} (Size {len(combinations)}): Logical Error Rate = {logical_error_rate:.2f}%")


--- 3-Round Global (Data + Ancillas) X-Error Rotations ---

Testing Rotation Angle: 0.42 rad (24.0°) | Projection Prob: 4.32%
  Dimension 1 (Size 17): Logical Error Rate = 0.00%
  Dimension 2 (Size 136): Logical Error Rate = 0.02%
  Dimension 3 (Size 680): Logical Error Rate = 0.07%
  Dimension 4 (Size 2380): Logical Error Rate = 0.14%
  Dimension 5 (Size 6188): Logical Error Rate = 0.24%
  Dimension 6 (Size 12376): Logical Error Rate = 0.35%
  Dimension 7 (Size 19448): Logical Error Rate = 0.49%
  Dimension 8 (Size 24310): Logical Error Rate = 0.65%
  Dimension 9 (Size 24310): Logical Error Rate = 0.82%
  Dimension 10 (Size 19448): Logical Error Rate = 1.02%
  Dimension 11 (Size 12376): Logical Error Rate = 1.23%
  Dimension 12 (Size 6188): Logical Error Rate = 1.46%
  Dimension 13 (Size 2380): Logical Error Rate = 1.70%
  Dimension 14 (Size 680): Logical Error Rate = 1.96%
  Dimension 15 (Size 136): Logical Error Rate = 2.24%
  Dimension 16 (Size 17): Logical Error Rate = 2.80%
  Dim

In [25]:
import stim
import numpy as np
import itertools
import pymatching

def generate_3_round_all_qubits_circuit(rotation_indices, rotation_angle_rad=0.0):
    """
    Generates a full 3-round surface code circuit (Round 0, 1, 2) + Final Readout.
    Accepts raw integer indices to inject coherent rotations across ANY data or ancilla qubits.
    """
    circuit = stim.Circuit()

    # --- QUBIT REGISTERS MAPPING ---
    def cq(i): return i
    def zq(i): return 9 + i
    def xq(i): return 13 + i
    all_xq = [xq(0), xq(1), xq(2), xq(3)]

    # Structural helper for the uniform syndrome extraction timeline
    def apply_syndrome_cycle(c):
        c.append("H", all_xq)
        
        # CNOT Layers 1-4
        c.append("CX", [cq(1), zq(1),  cq(3), zq(2),  cq(5), zq(3)])
        c.append("CX", [xq(1), cq(0),  xq(2), cq(4),  xq(3), cq(6)])
        c.append("CX", [cq(4), zq(1),  cq(6), zq(2),  cq(8), zq(3)])
        c.append("CX", [xq(1), cq(1),  xq(2), cq(5),  xq(3), cq(7)])
        
        # CNOT Layers 5-8
        c.append("CX", [cq(0), zq(0),  cq(2), zq(1),  cq(4), zq(2)])
        c.append("CX", [xq(0), cq(1),  xq(1), cq(3),  xq(2), cq(7)])
        c.append("CX", [cq(3), zq(0),  cq(5), zq(1),  cq(7), zq(2)])
        c.append("CX", [xq(0), cq(2),  xq(1), cq(4),  xq(2), cq(8)])
        c.append("H", all_xq)

    meas_targets = [xq(0), zq(0), xq(1), zq(1), xq(2), zq(2), xq(3), zq(3)]
    
    # ----------------------------------------------------
    # ROUND 0: Stable Baseline Initialization
    # ----------------------------------------------------
    apply_syndrome_cycle(circuit)
    circuit.append("M", meas_targets)
    circuit.append("R", meas_targets)
    
    # Initial deterministic Z-basis anchors
    circuit.append("DETECTOR", stim.target_rec(-7))  # zq0
    circuit.append("DETECTOR", stim.target_rec(-5))  # zq1
    circuit.append("DETECTOR", stim.target_rec(-3))  # zq2
    circuit.append("DETECTOR", stim.target_rec(-1))  # zq3

    # ----------------------------------------------------
    # ROUND 1: GLOBAL COHERENT ROTATIONS (DATA + ANCILLA) (Z-ERROR)
    # ----------------------------------------------------
    if rotation_indices and rotation_angle_rad > 0.0:
        # Calculate Born Rule projection probability: P = sin^2(theta / 2)
        projection_prob = np.sin(rotation_angle_rad / 2.0) ** 2
        # Target the raw input integer array directly
        circuit.append("Z_ERROR", rotation_indices, projection_prob)
        
    apply_syndrome_cycle(circuit)
    circuit.append("M", meas_targets)
    circuit.append("R", meas_targets)
    
    # Natively tracks ancilla fluctuations relative to Round 0
    for i in range(8):
        circuit.append("DETECTOR", [stim.target_rec(-8 + i), stim.target_rec(-16 + i)])

    # ----------------------------------------------------
    # ROUND 2: Verification Buffer Round
    # ----------------------------------------------------
    apply_syndrome_cycle(circuit)
    circuit.append("M", meas_targets)
    
    # Tracks stabilization relative to Round 1
    for i in range(8):
        circuit.append("DETECTOR", [stim.target_rec(-8 + i), stim.target_rec(-16 + i)])

    # ----------------------------------------------------
    # FINAL READOUT DATA DESTRUCTION BOUNDARY
    # ----------------------------------------------------
    data_targets = [cq(i) for i in range(9)]
    circuit.append("M", data_targets)

    # Closes out final stabilizer tracking frame (Data vs Round 2 Z-ancillas)
    circuit.append("DETECTOR", [stim.target_rec(-9), stim.target_rec(-6), stim.target_rec(-16)])  # zq0
    circuit.append("DETECTOR", [stim.target_rec(-8), stim.target_rec(-7), stim.target_rec(-5), stim.target_rec(-4), stim.target_rec(-14)])  # zq1
    circuit.append("DETECTOR", [stim.target_rec(-6), stim.target_rec(-5), stim.target_rec(-3), stim.target_rec(-2), stim.target_rec(-12)])  # zq2
    circuit.append("DETECTOR", [stim.target_rec(-4), stim.target_rec(-1), stim.target_rec(-10)])  # zq3

    circuit.append("OBSERVABLE_INCLUDE", [stim.target_rec(-9), stim.target_rec(-8), stim.target_rec(-7)], 0)

    return circuit

# --- MAIN EXECUTION BLOCK ---
if __name__ == "__main__":
    print("--- 3-Round Global (Data + Ancillas) Z-Error Rotations ---")
    
    # Total physical index map array: 9 data qubits + 4 Z-ancillas + 4 X-ancillas = 17 qubits total
    total_circuit_qubits = 17 
    shots_per_test = 1000 
    
    # Incremental steps sweep list configuration
    rotation_angles = np.linspace(0, 2 * np.pi, 16)[1:]
    
    # Initialize graph topology on ALL 17 elements to open edge definitions inside PyMatching
    base_circuit = generate_3_round_all_qubits_circuit(rotation_indices=list(range(total_circuit_qubits)), rotation_angle_rad=0.5)
    error_model = base_circuit.detector_error_model(decompose_errors=True)
    matching = pymatching.Matching.from_detector_error_model(error_model)
    
    # Multi-Angle Sweep Controller
    for angle_rad in rotation_angles:
        deg = np.degrees(angle_rad)
        proj_p = np.sin(angle_rad / 2.0) ** 2
        print(f"\n=========================================================================")
        print(f"Testing Rotation Angle: {angle_rad:.2f} rad ({deg:.1f}°) | Projection Prob: {proj_p*100:.2f}%")
        print(f"=========================================================================")
        
        # Inner dimension sweep expands to handle up to full 17 element set bounds
        for weight in range(1, total_circuit_qubits + 1):
            combinations = list(itertools.combinations(range(total_circuit_qubits), weight))
            total_failures_in_dimension = 0
            total_shots_in_dimension = len(combinations) * shots_per_test
            
            for rotation_set in combinations:
                # Convert tuple permutation block cleanly into raw integer list array for Stim
                test_circuit = generate_3_round_all_qubits_circuit(rotation_indices=list(rotation_set), rotation_angle_rad=angle_rad)
                
                sampler = test_circuit.compile_detector_sampler()
                det_samples, obs_samples = sampler.sample(shots=shots_per_test, separate_observables=True)
                
                predicted_observables = matching.decode_batch(det_samples)
                actual_observables = obs_samples[:, 0]
                
                shot_failures = np.sum(actual_observables != predicted_observables[:, 0])
                total_failures_in_dimension += shot_failures
                    
            logical_error_rate = (total_failures_in_dimension / total_shots_in_dimension) * 100
            print(f"  Dimension {weight} (Size {len(combinations)}): Logical Error Rate = {logical_error_rate:.2f}%")


--- 3-Round Global (Data + Ancillas) Z-Error Rotations ---

Testing Rotation Angle: 0.42 rad (24.0°) | Projection Prob: 4.32%
  Dimension 1 (Size 17): Logical Error Rate = 0.00%
  Dimension 2 (Size 136): Logical Error Rate = 0.00%
  Dimension 3 (Size 680): Logical Error Rate = 0.00%
  Dimension 4 (Size 2380): Logical Error Rate = 0.00%
  Dimension 5 (Size 6188): Logical Error Rate = 0.00%
  Dimension 6 (Size 12376): Logical Error Rate = 0.00%
  Dimension 7 (Size 19448): Logical Error Rate = 0.00%
  Dimension 8 (Size 24310): Logical Error Rate = 0.00%
  Dimension 9 (Size 24310): Logical Error Rate = 0.00%
  Dimension 10 (Size 19448): Logical Error Rate = 0.00%
  Dimension 11 (Size 12376): Logical Error Rate = 0.00%
  Dimension 12 (Size 6188): Logical Error Rate = 0.00%
  Dimension 13 (Size 2380): Logical Error Rate = 0.00%
  Dimension 14 (Size 680): Logical Error Rate = 0.00%
  Dimension 15 (Size 136): Logical Error Rate = 0.00%
  Dimension 16 (Size 17): Logical Error Rate = 0.00%
  Dim

In [26]:
import stim
import numpy as np
import itertools
import pymatching

def generate_3_round_all_qubits_circuit(rotation_indices, rotation_angle_rad=0.0):
    """
    Generates a full 3-round surface code circuit (Round 0, 1, 2) + Final Readout.
    Accepts raw integer indices to inject coherent rotations across ANY data or ancilla qubits.
    """
    circuit = stim.Circuit()

    # --- QUBIT REGISTERS MAPPING ---
    def cq(i): return i
    def zq(i): return 9 + i
    def xq(i): return 13 + i
    all_xq = [xq(0), xq(1), xq(2), xq(3)]

    # Structural helper for the uniform syndrome extraction timeline
    def apply_syndrome_cycle(c):
        c.append("H", all_xq)
        
        # CNOT Layers 1-4
        c.append("CX", [cq(1), zq(1),  cq(3), zq(2),  cq(5), zq(3)])
        c.append("CX", [xq(1), cq(0),  xq(2), cq(4),  xq(3), cq(6)])
        c.append("CX", [cq(4), zq(1),  cq(6), zq(2),  cq(8), zq(3)])
        c.append("CX", [xq(1), cq(1),  xq(2), cq(5),  xq(3), cq(7)])
        
        # CNOT Layers 5-8
        c.append("CX", [cq(0), zq(0),  cq(2), zq(1),  cq(4), zq(2)])
        c.append("CX", [xq(0), cq(1),  xq(1), cq(3),  xq(2), cq(7)])
        c.append("CX", [cq(3), zq(0),  cq(5), zq(1),  cq(7), zq(2)])
        c.append("CX", [xq(0), cq(2),  xq(1), cq(4),  xq(2), cq(8)])
        c.append("H", all_xq)

    meas_targets = [xq(0), zq(0), xq(1), zq(1), xq(2), zq(2), xq(3), zq(3)]
    
    # ----------------------------------------------------
    # ROUND 0: Stable Baseline Initialization
    # ----------------------------------------------------
    apply_syndrome_cycle(circuit)
    circuit.append("M", meas_targets)
    circuit.append("R", meas_targets)
    
    # Initial deterministic Z-basis anchors
    circuit.append("DETECTOR", stim.target_rec(-7))  # zq0
    circuit.append("DETECTOR", stim.target_rec(-5))  # zq1
    circuit.append("DETECTOR", stim.target_rec(-3))  # zq2
    circuit.append("DETECTOR", stim.target_rec(-1))  # zq3

    # ----------------------------------------------------
    # ROUND 1: GLOBAL COHERENT ROTATIONS (DATA + ANCILLA) (Y-ERROR)
    # ----------------------------------------------------
    if rotation_indices and rotation_angle_rad > 0.0:
        # Calculate Born Rule projection probability: P = sin^2(theta / 2)
        projection_prob = np.sin(rotation_angle_rad / 2.0) ** 2
        # Target the raw input integer array directly
        circuit.append("Y_ERROR", rotation_indices, projection_prob)
        
    apply_syndrome_cycle(circuit)
    circuit.append("M", meas_targets)
    circuit.append("R", meas_targets)
    
    # Natively tracks ancilla fluctuations relative to Round 0
    for i in range(8):
        circuit.append("DETECTOR", [stim.target_rec(-8 + i), stim.target_rec(-16 + i)])

    # ----------------------------------------------------
    # ROUND 2: Verification Buffer Round
    # ----------------------------------------------------
    apply_syndrome_cycle(circuit)
    circuit.append("M", meas_targets)
    
    # Tracks stabilization relative to Round 1
    for i in range(8):
        circuit.append("DETECTOR", [stim.target_rec(-8 + i), stim.target_rec(-16 + i)])

    # ----------------------------------------------------
    # FINAL READOUT DATA DESTRUCTION BOUNDARY
    # ----------------------------------------------------
    data_targets = [cq(i) for i in range(9)]
    circuit.append("M", data_targets)

    # Closes out the final stabilizer tracking frame (Data vs Round 2 Z-ancillas)
    circuit.append("DETECTOR", [stim.target_rec(-9), stim.target_rec(-6), stim.target_rec(-16)])  # zq0
    circuit.append("DETECTOR", [stim.target_rec(-8), stim.target_rec(-7), stim.target_rec(-5), stim.target_rec(-4), stim.target_rec(-14)])  # zq1
    circuit.append("DETECTOR", [stim.target_rec(-6), stim.target_rec(-5), stim.target_rec(-3), stim.target_rec(-2), stim.target_rec(-12)])  # zq2
    circuit.append("DETECTOR", [stim.target_rec(-4), stim.target_rec(-1), stim.target_rec(-10)])  # zq3

    circuit.append("OBSERVABLE_INCLUDE", [stim.target_rec(-9), stim.target_rec(-8), stim.target_rec(-7)], 0)

    return circuit

# --- MAIN EXECUTION BLOCK ---
if __name__ == "__main__":
    print("--- 3-Round Global (Data + Ancillas) Y-Error Rotations ---")
    
    # Total physical index map array: 9 data qubits + 4 Z-ancillas + 4 X-ancillas = 17 qubits total
    total_circuit_qubits = 17 
    shots_per_test = 1000 
    
    # Incremental steps sweep list configuration
    rotation_angles = np.linspace(0, 2 * np.pi, 16)[1:]
    
    # Initialize graph topology on ALL 17 elements to open edge definitions inside PyMatching
    base_circuit = generate_3_round_all_qubits_circuit(rotation_indices=list(range(total_circuit_qubits)), rotation_angle_rad=0.5)
    error_model = base_circuit.detector_error_model(decompose_errors=True, ignore_decomposition_failures=True)
    matching = pymatching.Matching.from_detector_error_model(error_model)
    
    # Multi-Angle Sweep Controller
    for angle_rad in rotation_angles:
        deg = np.degrees(angle_rad)
        proj_p = np.sin(angle_rad / 2.0) ** 2
        print(f"\n=========================================================================")
        print(f"Testing Rotation Angle: {angle_rad:.2f} rad ({deg:.1f}°) | Projection Prob: {proj_p*100:.2f}%")
        print(f"=========================================================================")
        
        # Inner dimension sweep expands to handle up to the full 17 element set bounds
        for weight in range(1, total_circuit_qubits + 1):
            combinations = list(itertools.combinations(range(total_circuit_qubits), weight))
            total_failures_in_dimension = 0
            total_shots_in_dimension = len(combinations) * shots_per_test
            
            for rotation_set in combinations:
                # Convert tuple permutation block cleanly into raw integer list array for Stim
                test_circuit = generate_3_round_all_qubits_circuit(rotation_indices=list(rotation_set), rotation_angle_rad=angle_rad)
                
                sampler = test_circuit.compile_detector_sampler()
                det_samples, obs_samples = sampler.sample(shots=shots_per_test, separate_observables=True)
                
                predicted_observables = matching.decode_batch(det_samples)
                actual_observables = obs_samples[:, 0]
                
                shot_failures = np.sum(actual_observables != predicted_observables[:, 0])
                total_failures_in_dimension += shot_failures
                    
            logical_error_rate = (total_failures_in_dimension / total_shots_in_dimension) * 100
            print(f"  Dimension {weight} (Size {len(combinations)}): Logical Error Rate = {logical_error_rate:.2f}%")


--- 3-Round Global (Data + Ancillas) Y-Error Rotations ---

Testing Rotation Angle: 0.42 rad (24.0°) | Projection Prob: 4.32%
  Dimension 1 (Size 17): Logical Error Rate = 0.23%
  Dimension 2 (Size 136): Logical Error Rate = 0.50%
  Dimension 3 (Size 680): Logical Error Rate = 0.77%
  Dimension 4 (Size 2380): Logical Error Rate = 1.03%
  Dimension 5 (Size 6188): Logical Error Rate = 1.27%
  Dimension 6 (Size 12376): Logical Error Rate = 1.52%
  Dimension 7 (Size 19448): Logical Error Rate = 1.78%
  Dimension 8 (Size 24310): Logical Error Rate = 2.03%
  Dimension 9 (Size 24310): Logical Error Rate = 2.29%
  Dimension 10 (Size 19448): Logical Error Rate = 2.54%
  Dimension 11 (Size 12376): Logical Error Rate = 2.81%
  Dimension 12 (Size 6188): Logical Error Rate = 3.06%
  Dimension 13 (Size 2380): Logical Error Rate = 3.28%
  Dimension 14 (Size 680): Logical Error Rate = 3.54%
  Dimension 15 (Size 136): Logical Error Rate = 3.79%
  Dimension 16 (Size 17): Logical Error Rate = 4.09%
  Dim